In [28]:
from skimage import io, measure
import numpy as np
from cell_paint_seg.utils import (
    get_id_to_path,
    get_id_from_name_96,
    check_valid_labels,
    label_celltype,
)
from tqdm.notebook import tqdm
import napari

In [29]:
channels = ["ER", "DNA", "Mito", "Actin", "RNA", "Golgi/membrane"]

# Save rgb

In [ ]:
path_r = "/Users/thomasathey/Documents/shavit-lab/fraenkel/papers/miccai/rebuttal/data/og/Experiment4_DB_AD_unmixed_zprojection_s144c4.tif"
path_g = "/Users/thomasathey/Documents/shavit-lab/fraenkel/papers/miccai/rebuttal/data/og/Experiment4_DB_AD_unmixed_zprojection_s144c5.tif"
path_b = "/Users/thomasathey/Documents/shavit-lab/fraenkel/papers/miccai/rebuttal/data/og/Experiment4_DB_AD_unmixed_zprojection_s144c2.tif"

img_r = io.imread(path_r)
img_g = io.imread(path_g)
img_b = io.imread(path_b)

img_r = np.amax(img_r, -1)
img_g = np.amax(img_g, -1)
img_b = np.amax(img_b, -1)

img_rgb = np.stack([img_r, img_g, img_b], axis=-1)
#io.imsave("/Users/thomasathey/Documents/shavit-lab/fraenkel/papers/miccai/rebuttal/data/3channel/e4_s144.tif", img_rgb)


# Clean up labels

In [58]:
dir_seg = (
    "/Users/thomasathey/Documents/shavit-lab/fraenkel/papers/miccai/rebuttal/data/3channel"
)

factor = 1 # 0.6

id_to_path_seg = get_id_to_path(dir_seg, tag=".png", id_from_name_nchar=7)

for id, paths in id_to_path_seg.items():
    for path in paths:
        if "nuc" in str(path):
            seg_nuc = io.imread(path)
        elif "soma" in str(path):
            seg_soma = io.imread(path)
        elif "cell" in str(path):
            seg_cell = io.imread(path)

    print("Separating components...")
    for comp, seg in zip(["Nuc", "Soma", "Cell"], [seg_nuc, seg_soma, seg_cell]):
        new_label = np.amax(seg) + 1
        for label in np.unique(seg):
            # background
            if label == 0:
                continue

            lbl = measure.label(seg == label)
            if not np.amax(lbl) == 1:
                regprops = measure.regionprops(lbl)
                print(
                    f"{comp} {label} has {np.amax(lbl)} disconnected components - renaming {[np.multiply(props.centroid, factor) for props in regprops]}..."
                )
                for props in regprops[1:]:
                    mask = lbl == props["label"]
                    seg[mask] = new_label
                    new_label += 1

    print("Matching somas to nuclei...")
    soma_to_nuc = {}
    for nuc_label in tqdm(np.unique(seg_nuc)):
        if nuc_label == 0:
            continue
        found = False
        for soma_label in np.unique(seg_soma):
            if soma_label == 0:
                continue
            recall = np.sum((seg_nuc == nuc_label) & (seg_soma == soma_label)) / np.sum(
                (seg_nuc == nuc_label)
            )

            if recall > 0.8:
                assert soma_label not in soma_to_nuc.keys()
                soma_to_nuc[soma_label] = nuc_label
                found = True
        if not found:
            wher = np.where(seg_nuc == nuc_label)
            print(
                f"nuc {nuc_label} not found in soma {(wher[0][0]*factor, wher[1][0]*factor)}"
            )
    
    seg_soma_relabel = np.zeros_like(seg_soma)
    for soma_label, nuc_label in soma_to_nuc.items():
        seg_soma_relabel[seg_soma == soma_label] = nuc_label
    seg_soma = seg_soma_relabel

    print("Matching somas to cells...")
    cell_to_soma = {}
    for soma_label in tqdm(np.unique(seg_soma)):
        if soma_label == 0:
            continue
        found = False
        for cell_label in np.unique(seg_cell):
            if cell_label == 0:
                continue
            recall = np.sum(
                (seg_soma == soma_label) & (seg_cell == cell_label)
            ) / np.sum((seg_soma == soma_label))

            if recall > 0.8:
                if cell_label in cell_to_soma.keys():
                    print(f"cell {cell_label} already matched to soma")
                # assert cell_label not in cell_to_soma.keys()
                cell_to_soma[cell_label] = soma_label
                found = True

        if not found:
            wher = np.where(seg_soma == soma_label)
            print(
                f"soma {soma_label} not found in cells {(wher[0][0]*factor, wher[1][0]*factor)}"
            )

    seg_cell_relabel = np.zeros_like(seg_cell)
    for cell_label, soma_label in cell_to_soma.items():
        seg_cell_relabel[seg_cell == cell_label] = soma_label
    seg_cell = seg_cell_relabel

    print("Subsetting...")
    for label in np.unique(seg_nuc):
        if label == 0:
            continue
        seg_soma[seg_nuc == label] = label
        seg_cell[seg_soma == label] = label

    print("Relabelling consecutively...")
    seg_nuc_relabeled = np.zeros_like(seg_nuc)
    seg_soma_relabeled = np.zeros_like(seg_soma)
    seg_cell_relabeled = np.zeros_like(seg_cell)
    counter = 1
    for label in np.unique(seg_nuc):
        if label == 0:
            continue
        seg_nuc_relabeled[seg_nuc == label] = counter
        seg_soma_relabeled[seg_soma == label] = counter
        seg_cell_relabeled[seg_cell == label] = counter
        counter += 1

    
    seg_nuc = seg_nuc_relabeled.astype(np.int32)
    seg_soma = seg_soma_relabeled.astype(np.int32)
    seg_cell = seg_cell_relabeled.astype(np.int32)

    out_dir = paths[0].parent
    io.imsave(out_dir / f"{id}_seg_nuc_relabelled.png", seg_nuc)
    io.imsave(out_dir / f"{id}_seg_soma_relabelled.png", seg_soma)
    io.imsave(out_dir / f"{id}_seg_cell_relabelled.png", seg_cell)

Separating components...
Matching somas to nuclei...


  0%|          | 0/242 [00:00<?, ?it/s]

Matching somas to cells...


  0%|          | 0/242 [00:00<?, ?it/s]

Subsetting...
Relabelling consecutively...


...

  0%|          | 0/151 [00:00<?, ?it/s]

Matching somas to cells...


  0%|          | 0/151 [00:00<?, ?it/s]

Subsetting...
Relabelling consecutively...


/var/folders/gy/jk_d3cx54vj18w9sm6x3sg_80000gn/T/ipykernel_21547/2381142023.py:122: UserWarning: /Users/thomasathey/Documents/shavit-lab/fraenkel/papers/miccai/rebuttal/data/3channel/e1_s036_seg_nuc_relabelled.png is a low contrast image
  io.imsave(out_dir / f"{id}_seg_nuc_relabelled.png", seg_nuc)
/var/folders/gy/jk_d3cx54vj18w9sm6x3sg_80000gn/T/ipykernel_21547/2381142023.py:123: UserWarning: /Users/thomasathey/Documents/shavit-lab/fraenkel/papers/miccai/rebuttal/data/3channel/e1_s036_seg_soma_relabelled.png is a low contrast image
  io.imsave(out_dir / f"{id}_seg_soma_relabelled.png", seg_soma)
/var/folders/gy/jk_d3cx54vj18w9sm6x3sg_80000gn/T/ipykernel_21547/2381142023.py:124: UserWarning: /Users/thomasathey/Documents/shavit-lab/fraenkel/papers/miccai/rebuttal/data/3channel/e1_s036_seg_cell_relabelled.png is a low contrast image
  io.imsave(out_dir / f"{id}_seg_cell_relabelled.png", seg_cell)


In [48]:
viewer = napari.Viewer()
viewer.add_labels(seg_nuc, name=f"{id}: nuc")
viewer.add_labels(seg_soma, name=f"{id}: soma")
viewer.add_labels(seg_cell, name=f"{id}: cell")

<Labels layer 'e1_s036: cell' at 0x182546c70>

# Evaulate alg segmentation